# 구글 드라이브 연결 및 데이터 불러오기

In [ ]:
# 본인 구글드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd drive/MyDrive/'Colab Notebooks'/데이터마이닝/

In [ ]:
import numpy as np
import pandas as pd
import re

# 데이터 불러오기
data=pd.read_csv('chatgpt_paraphrases.csv')

In [ ]:
data.head()
data.dtypes
data

# 데이터 전처리 및 테스트 셋 분류

In [ ]:
# 필요한 데이터 뽑아내기 (Collecting Nescessary Data)
category={}
for i in range(len(data)):
    chatgpt=data.iloc[i]["paraphrases"][1:-1].split(', ')
    for j in chatgpt[:1]:
        category[j[1:-1]]='chatgpt'
    category[data.iloc[i]['text']]="human"

# 데이터프레임 형식으로 바꾸기 (Converting Dictionary
data=pd.DataFrame(category.items(),columns=["text","category"])
data=data.sample(frac=1)

data

data["category"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 테스트 셋 분류
X=data['text']
y=data['category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Tfidf를 사용하여 벡터화 (Vectorizing Using Tfidf)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 모델 선정

모델 후보들

KNN(K Nearest Neighbor),
SVC(Support Vector Machines),
RFC(Random Forest Classifier),
DTC(Decision Tree Classifier)

를 사용할 계획이다.

## KNN(K Nearest Neighbor)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11], 'weights': ['uniform', 'distance']}
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    knn_model = KNeighborsClassifier(**params)
    knn_model.fit(X_train_tfidf, y_train)
    y_pred = knn_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


n_neighbors 숫자가 커질수록 mcc 값이 커지는것을 확인

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'n_neighbors': list(range(1,101)), 'weights': ['uniform', 'distance']}
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    knn_model = KNeighborsClassifier(**params)
    knn_model.fit(X_train_tfidf, y_train)
    y_pred = knn_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


Best Hyperparameters: {'n_neighbors': 43, 'weights': 'distance'}

Evaluation Metrics:
Accuracy: 0.72
Precision: 0.7218863015823609
Recall: 0.72
F1 Score: 0.7190533965955703
Matthews Correlation Coefficient: 0.4413253711772089

## SVC(Support Vector Machines)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'C': [0.01, 0.1, 1, 10, 100, 1000], 'kernel': ['linear', 'rbf']}
svc = SVC()
grid_search = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    svc_model = SVC(**params)
    svc_model.fit(X_train_tfidf, y_train)
    y_pred = svc_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


Best Hyperparameters: {'C': 10, 'kernel': 'rbf'}

Evaluation Metrics:
Accuracy: 0.7625
Precision: 0.7636107310458006
Recall: 0.7625
F1 Score: 0.7619170211585666
Matthews Correlation Coefficient: 0.5253533759188264

## RFC(Random Forest Classifier)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
rfc = RandomForestClassifier()
grid_search = GridSearchCV(rfc, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    rfc_model = RandomForestClassifier(**params)
    rfc_model.fit(X_train_tfidf, y_train)
    y_pred = rfc_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_rfc = grid_search.best_estimator_
y_pred = best_rfc.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


## DTC(Decision Tree Classifier)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'criterion': ['gini', 'entropy'], 'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
dtc = DecisionTreeClassifier()
grid_search = GridSearchCV(dtc, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    dtc_model = DecisionTreeClassifier(**params)
    dtc_model.fit(X_train_tfidf, y_train)
    y_pred = dtc_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_dtc = grid_search.best_estimator_
y_pred = best_dtc.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


# 최종 선택한 모델 교차 검증 수행

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Hyperparameter tuning using GridSearchCV
param_grid = {'n_estimators': [200], 'max_depth': [None], 'min_samples_split': [5], 'min_samples_leaf': [4]}
rfc = RandomForestClassifier()
grid_search = GridSearchCV(rfc, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

# Print the results for each hyperparameter combination
results = grid_search.cv_results_
for mean_score, params in zip(results['mean_test_score'], results['params']):
    rfc_model = RandomForestClassifier(**params)
    rfc_model.fit(X_train_tfidf, y_train)
    y_pred = rfc_model.predict(X_test_tfidf)

    # Calculate precision, recall, f1-score, and Matthews correlation coefficient
    pre = metrics.precision_score(y_test, y_pred, average='weighted')
    rec = metrics.recall_score(y_test, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    mcc = metrics.matthews_corrcoef(y_test, y_pred)

    print(f"Hyperparameters: {params}")
    print("  Accuracy:", mean_score)
    print("  Precision:", pre)
    print("  Recall:", rec)
    print("  F1 Score:", f1)
    print("  Matthews Correlation Coefficient:", mcc)
    print("-----------")

# Print the best hyperparameters
print("\nBest Hyperparameters:", grid_search.best_params_)

# Evaluate the model with the best hyperparameters on the test set
best_rfc = grid_search.best_estimator_
y_pred = best_rfc.predict(X_test_tfidf)

# Calculate accuracy, precision, recall, f1-score, and Matthews correlation coefficient
acc = metrics.accuracy_score(y_test, y_pred)
pre = metrics.precision_score(y_test, y_pred, average='weighted')
rec = metrics.recall_score(y_test, y_pred, average='weighted')
f1 = metrics.f1_score(y_test, y_pred, average='weighted')
mcc = metrics.matthews_corrcoef(y_test, y_pred)

# Print the evaluation metrics
print("\nEvaluation Metrics:")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1 Score:", f1)
print("Matthews Correlation Coefficient:", mcc)


In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# Define the selected Random Forest Classifier with the best hyperparameters
best_rfc = RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_split=5, min_samples_leaf=1)

# Define the number of folds for K-fold cross-validation
n_folds = 5

# Create a KFold object
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Perform K-fold cross-validation
cv_results = cross_val_score(best_rfc, X_train_tfidf, y_train, cv=kf, scoring='accuracy')

# Print the accuracy for each fold
for i, acc in enumerate(cv_results, start=1):
    print(f"Fold {i} Accuracy: {acc:.3f}")

# Print the average accuracy across all folds
print(f"\nAverage Accuracy across {n_folds} Folds: {cv_results.mean():.3f}")


In [ ]:
rfc.fit(X_train_tfidf,y_train)

# 결과 Confusion & ROC

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred =rfc.predict(X_test_tfidf)
cm = confusion_matrix(y_test, y_pred)
print(cm)
y_test.value_counts()

import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
df_cm = pd.DataFrame(cm, index = [i for i in ["ChatGPT","Human"]],
                  columns = [i for i in ["ChatGPT","Human"]])
plt.figure(figsize = (10,7))
sn.heatmap(df_cm, annot=True,cmap="YlGnBu", fmt='g')

In [ ]:
from sklearn.metrics import roc_curve,auc
y_prob = rfc.predict_proba(X_test_tfidf)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob, pos_label='human')

# Calculate the area under the ROC curve
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# 사용해보기

In [ ]:
def predict_text_category(model, text):
    text_vectorized = vectorizer.transform([text])
    prediction_prob = model.predict_proba(text_vectorized)
    predicted_class_idx = np.argmax(prediction_prob)
    unique_class_labels = np.unique(y_train)
    predicted_category = unique_class_labels[predicted_class_idx]
    return predicted_category

In [ ]:
text_to_predict = "Waves pound across a stone jetty. A MAN sits fishing whilehis young son, BRANDO strolls toward the open sea. He pokesat rocks and seaweed with a fishing pole. He glances down atSomething wedged between the rocks beneath his feet. He pokesat it."
predicted_category = predict_text_category(rfc, text_to_predict)
print("Predicted Category:", predicted_category)

In [ ]:
text_to_predict = "Arthur brushes past Mal, shaking his head. She nears Cobb. Looks out at the DROP."
predicted_category = predict_text_category(rfc, text_to_predict)
print("Predicted Category:", predicted_category)

In [ ]:
text_to_predict = "A script is a list of programmatically-written instructions that can be carried out on command."
predicted_category = predict_text_category(rfc, text_to_predict)
print("Predicted Category:", predicted_category)

In [ ]:
text_to_predict = "A controller is an individual who has responsibility for all accounting-related activities, including high-level accounting, managerial accounting, and finance activities, within a company."
predicted_category = predict_text_category(rfc, text_to_predict)
print("Predicted Category:", predicted_category)